## Installation (execute only if running on a cloud platform!)¶

In [ ]:
# -- Use the following line for google colab or Binder
#! pip install -q 'corner==2.0.1' 'bilby==1.0.4' 'astropy==4.0.3'

**Important**: With Google Colab, you may need to restart the runtime after running the cell above.

## Initialization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import corner
import os
from scipy.stats import gaussian_kde

## Get the data

Selecting the event, let's pick GW150914.

In [ ]:
label = 'GW150914'

# if you do not have wget installed, simply download manually 
# https://dcc.ligo.org/LIGO-P1800370/public/GW150914_GWTC-1.hdf5 
# from your browser
! wget https://dcc.ligo.org/LIGO-P1800370/public/{label}_GWTC-1.hdf5

In [ ]:
posterior_file = './'+label+'_GWTC-1.hdf5'
posterior = h5py.File(posterior_file, 'r')
os.remove('GW150914_GWTC-1.hdf5')

### Looking into the file structure

In [ ]:
print('This file contains four datasets: ',posterior.keys())

This data file contains several datasets, two using separate models for the gravitational waveform (`IMRPhenomPv2` and `SEOBNRv3` respectively, see the [paper](https://arxiv.org/abs/1811.12907) for more details). 

It also contiains a joint dataset, combining equal numbers of samples from each individual model, these datasets are what is shown in the [paper](https://arxiv.org/abs/1811.12907). 

Finally, there is a dataset containing samples drawn from the prior used for the analyses.

In [ ]:
print(posterior['Overall_posterior'].dtype.names)

Here are some brief descriptions of these parameters and their uses:

 * `luminosity_distance_Mpc`: luminosity distance [Mpc]

 * `m1_detector_frame_Msun`: primary (larger) black hole mass (detector frame) [solar mass]

 * `m2_detector_frame_Msun`: secondary (smaller) black hole mass (detector frame) [solar mass]

 * `right_ascension`, `declination`: right ascension and declination of the source [rad].

 * `costheta_jn`: cosine of the angle between line of sight and total angular momentum vector of system.

 * `spin1`, `costilt1`: primary (larger) black hole spin magnitude (dimensionless) and cosine of the zenith angle between the spin and the orbital angular momentum vector of system.

 * `spin2`, `costilt2`: secondary (smaller) black hole spin magnitude (dimensionless) and cosine of the zenith angle between the spin and the orbital angular momentum vector of system.

A convenient (and pretty) way to load up this array of samples is to use [pandas](https://pandas.pydata.org/):

In [ ]:
samples=pd.DataFrame.from_records(np.array(posterior['Overall_posterior']))

In [ ]:
samples

Those are all the samples stored in the `Overall` dataset. 

### Plotting

We can plot all of them with, for instance, the [corner](https://corner.readthedocs.io/en/latest/) package:

In [ ]:
corner.corner(samples,labels=['costhetajn',
                                'distance [Mpc]',
                                'ra [deg]',
                                'dec [deg]',
                                'mass1 [Msun]',
                                'mass2 [Msun]',
                                'spin1',
                                'spin2',
                                'costilt1',
                                'costilt2']);

Each one and two dimentional histogram are *marginalised* probability density functions. We can manualy select one parameter, say `luminosity distance`, and plot the four different marginalised distributions:

In [ ]:
plt.hist(posterior['prior']['luminosity_distance_Mpc'], bins = 100, label='prior', alpha=0.8, density=True)
plt.hist(posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc'], bins = 100, label='IMRPhenomPv2 posterior', alpha=0.8, density=True)
plt.hist(posterior['SEOBNRv3_posterior']['luminosity_distance_Mpc'], bins = 100, label='SEOBNRv3 posterior', alpha=0.8, density=True)
plt.hist(posterior['Overall_posterior']['luminosity_distance_Mpc'], bins = 100, label='Overall posterior', alpha=0.8, density=True)
plt.xlabel(r'$D_L (Mpc)$')
plt.ylabel('Probability Density Function')
plt.legend()
plt.show()

Sometimes we might want to remove a prior from the posterior. Let us assume that we want to remove the $d_L^2$ prior from the luminosity distance estimation. There are several ways to do it numerically. Below we will show two methologies

In [ ]:
''' 
One possibility is to re-sample from the posterior samples, but applying a weight inversely proportional to the prior used
'''
weight=1./posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc']**2. # We define a weight inversely proportional to the prior
weight/=weight.sum()
newd=np.random.choice(posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc'],size=5000,p=weight) # We resample the luminosity distance taking into account the weights
_=plt.hist(newd,bins='auto',density=True,histtype='step',label='Prior removed')
_=plt.hist(posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc'],bins='auto',density=True,histtype='step',label='With prior')
plt.legend()
plt.xlabel(r'$d_L$[Mpc]')
plt.ylabel(r'PDF [Mpc$^{-1}$]')

In [ ]:
''' 
Another possibility is to use a gaussian kernel density estimate from Scipy. This is a function that uses
a gaussian kernel to estimate a probability function. It can take as input also some weights that we punt inversely proportional to the prior
'''
kde=gaussian_kde(posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc'],weights=posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc']**-2.)
dl_eval=np.linspace(0,800,100)
plt.plot(dl_eval,kde(dl_eval),label='KDE no prior')
_=plt.hist(posterior['IMRPhenomPv2_posterior']['luminosity_distance_Mpc'],bins='auto',density=True,histtype='step',label='With prior')
plt.legend()
plt.xlabel(r'$d_L$[Mpc]')
plt.ylabel(r'PDF [Mpc$^{-1}$]')

### Computing new quantities

The masses given are the ones measured at the detector, i.e. in the *detector frame*. To get the actual (*source frame*) masses of the source black holes, we need to correct for the cosmological redshift of the gravitational wave. This forces us to assume a cosmology:

In [ ]:
import astropy.units as u
from astropy.cosmology import Planck15, z_at_value

We now compute the redshift value for all the samples (using only their distance value). See [astropy.cosmology](http://docs.astropy.org/en/stable/api/astropy.cosmology.z_at_value.html) for implementation details, in particular how to make the following more efficient:

In [ ]:
z = np.array([z_at_value(Planck15.luminosity_distance, dist * u.Mpc) for dist in samples['luminosity_distance_Mpc']])

In [ ]:
samples['m1_source_frame_Msun']=samples['m1_detector_frame_Msun']/(1.0+z)
samples['m2_source_frame_Msun']=samples['m2_detector_frame_Msun']/(1.0+z)
samples['redshift']=z

And we can plot the marginalised probability density functions:

In [ ]:
corner.corner(samples[['m1_source_frame_Msun','m2_source_frame_Msun','redshift']],labels=['m1 (source) [MSun]',
                                                                                          'm2 (source) [MSun]',
                                                                                          'z']);

## Challenge question

Challenge: Reweight the luminosity distance posterior samples of GW150914 using a uniform in comoving volume prior. 

Tips: they are in the slides if you need them.

Advanced: We actually did something wrong, when going from $d_L$ to $z$ we need to take into account for a Jacobian from the change of variables. Can you compute the Jacobian within a flat $\Lambda$ CDM cosmology and tell why this is not important for GW150914?

## Exploring other events

pesummary allows you to explore other events. You can pick some of your favorite ones from https://gwosc.org/eventapi/html/GWTC/

In [ ]:
from pesummary.gw.fetch import fetch_open_samples

data = fetch_open_samples("GW200105_162426")
print(data.labels)
samples = data.samples_dict[data.labels[0]]